# Step 7 — Machine Learning Support: Home Credit Default Risk

Notebook Kaggle-ready này dùng output Step 4:

- `final_customer_analysis_train.csv.gz`
- `final_customer_analysis_test.csv.gz`
- `sample_submission.csv` nếu muốn tạo submission.

Mục tiêu:

1. Chạy full-data benchmark cho binary classification.
2. Chạy đủ các model ML cơ bản có thể đánh giá hợp lệ trên customer-level binary target.
3. Chạy thêm các recipe học từ Home Credit top solutions: LightGBM regularized, XGBoost, CatBoost, ensemble.
4. Mọi model chạy được đều có đầy đủ metric: ROC-AUC, PR-AUC, log loss, Brier, KS, confusion matrix, accuracy, precision, recall, specificity, F1, MCC, Lift@TopK, threshold analysis, calibration.
5. Xuất bảng, hình, feature importance, SHAP và `submission.csv`.


## Lưu ý khoa học

Notebook chạy **full rows**, không sample dòng để train benchmark.

Một số model như SVM/KNN/Naive Bayes/MLP không scale tốt với full one-hot matrix. Để vẫn chạy full data, notebook dùng biểu diễn compact:

- full rows,
- one-hot,
- SelectKBest,
- TruncatedSVD,
- StandardScaler.

Các model như Cox survival/discrete-time hazard cần time-to-event/person-period data nên được ghi vào applicability table thay vì ép chạy sai dữ liệu.


In [ ]:
# ============================================================
# 1. Environment Setup
# ============================================================
import os, sys, subprocess, importlib.util, warnings, time, gc, json, math
from pathlib import Path
warnings.filterwarnings("ignore")

KAGGLE = Path("/kaggle").exists()
OUTPUT_DIR = Path("/kaggle/working/step7_ml_outputs") if KAGGLE else Path("step7_ml_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "lightgbm": "lightgbm",
    "xgboost": "xgboost",
    "catboost": "catboost",
    "shap": "shap",
    "category_encoders": "category_encoders",
    "joblib": "joblib",
}

def ensure_package(import_name, pip_name):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"OK: {import_name}")

for import_name, pip_name in REQUIRED.items():
    ensure_package(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import expit
from scipy.stats import ks_2samp

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, log_loss, matthews_corrcoef, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MaxAbsScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier

from lightgbm import LGBMClassifier
from lightgbm import early_stopping as lgb_early_stopping, log_evaluation as lgb_log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
import category_encoders as ce
import shap
import joblib

RANDOM_STATE = 42
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

USE_GPU = True
RUN_EXPENSIVE_MODELS = True
RUN_RESEARCH_LGBM_RECIPES = True
RUN_CV_TOP_MODELS = True
RUN_SHAP = True

TEST_SIZE = 0.20
N_FOLDS = 5
TOP_FEATURES_FOR_COMPACT_MODELS = 120
SVD_COMPONENTS = 80
EARLY_STOPPING_ROUNDS = 200

np.random.seed(RANDOM_STATE)
print("Output dir:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Load Full Dataset
# ============================================================
def find_input_file(filename):
    for root in [Path("/kaggle/input"), Path(".")]:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Cannot find {filename}. Upload it to a Kaggle Dataset.")

TRAIN_PATH = find_input_file("final_customer_analysis_train.csv.gz")
TEST_PATH = find_input_file("final_customer_analysis_test.csv.gz")
try:
    SAMPLE_SUB_PATH = find_input_file("sample_submission.csv")
except FileNotFoundError:
    SAMPLE_SUB_PATH = None

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("SAMPLE_SUB_PATH:", SAMPLE_SUB_PATH)

t0 = time.time()
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"Load time  : {(time.time()-t0)/60:.2f} minutes")

assert TARGET in train.columns
assert ID_COL in train.columns
assert ID_COL in test.columns

schema_report = pd.DataFrame([
    {"dataset": "train", "rows": len(train), "columns": train.shape[1], "default_rate": train[TARGET].mean(), "duplicate_ids": train[ID_COL].duplicated().sum()},
    {"dataset": "test", "rows": len(test), "columns": test.shape[1], "default_rate": np.nan, "duplicate_ids": test[ID_COL].duplicated().sum()},
])
display(schema_report)
schema_report.to_csv(OUTPUT_DIR / "01_schema_report.csv", index=False)

train_cols_no_target = [c for c in train.columns if c != TARGET]
for c in sorted(set(train_cols_no_target) - set(test.columns)):
    test[c] = np.nan
test = test[train_cols_no_target]


In [ ]:
# ============================================================
# 3. ML Feature Engineering
# ============================================================
def add_ml_features(df):
    df = df.copy()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    ext_cols = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in df.columns]
    if ext_cols:
        df["ML_EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
        df["ML_EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
        df["ML_EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)
        df["ML_EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)
        df["ML_EXT_SOURCE_MISSING_COUNT"] = df[ext_cols].isna().sum(axis=1)

    def safe_ratio(num, den, name):
        if num in df.columns and den in df.columns and name not in df.columns:
            df[name] = df[num] / df[den].replace(0, np.nan)

    safe_ratio("AMT_CREDIT", "AMT_INCOME_TOTAL", "ML_CREDIT_INCOME_RATIO")
    safe_ratio("AMT_ANNUITY", "AMT_INCOME_TOTAL", "ML_ANNUITY_INCOME_RATIO")
    safe_ratio("AMT_GOODS_PRICE", "AMT_CREDIT", "ML_GOODS_CREDIT_RATIO")
    safe_ratio("AMT_ANNUITY", "AMT_CREDIT", "ML_ANNUITY_CREDIT_RATIO")

    signal_cols = [c for c in df.columns if c not in [ID_COL, TARGET]]
    df["ML_MISSING_COUNT"] = df[signal_cols].isna().sum(axis=1)
    df["ML_MISSING_RATE"] = df["ML_MISSING_COUNT"] / max(len(signal_cols), 1)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return df

train_fe = add_ml_features(train)
test_fe = add_ml_features(test)

feature_cols = [c for c in train_fe.columns if c != TARGET]
for c in feature_cols:
    if c not in test_fe.columns:
        test_fe[c] = np.nan
test_fe = test_fe[feature_cols]

y = train_fe[TARGET].astype(int)
ids_test = test_fe[ID_COL].copy()
X = train_fe.drop(columns=[TARGET, ID_COL])
X_test_final = test_fe.drop(columns=[ID_COL])

constant_cols = X.nunique(dropna=False)[lambda s: s <= 1].index.tolist()
print("Constant columns dropped:", len(constant_cols))
X = X.drop(columns=constant_cols)
X_test_final = X_test_final.drop(columns=[c for c in constant_cols if c in X_test_final.columns])

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Feature count:", X.shape[1])
print("Numeric:", len(num_cols), "Categorical:", len(cat_cols))
print("Default rate:", y.mean())

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
print("Train split:", X_train.shape, "Validation split:", X_val.shape)


In [ ]:
# ============================================================
# 4. Metrics
# ============================================================
def safe_proba_from_model(model, X_data):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_data)
        if isinstance(proba, list):
            proba = proba[0]
        if getattr(proba, "ndim", 1) == 2:
            return np.asarray(proba[:, 1], dtype=float), "predict_proba"
        return np.asarray(proba, dtype=float), "predict_proba_1d"
    if hasattr(model, "decision_function"):
        score = np.asarray(model.decision_function(X_data), dtype=float)
        return expit(score), "sigmoid_decision_function"
    pred = model.predict(X_data)
    return np.asarray(pred, dtype=float), "hard_prediction_only"

def ks_statistic(y_true, score):
    y_true, score = np.asarray(y_true).astype(int), np.asarray(score)
    return float(ks_2samp(score[y_true == 1], score[y_true == 0]).statistic)

def top_percentile_metrics(y_true, score, pct):
    y_true, score = np.asarray(y_true).astype(int), np.asarray(score)
    k = max(1, int(np.ceil(len(y_true) * pct)))
    idx = np.argsort(-score)[:k]
    default_top = y_true[idx].sum()
    base = y_true.mean()
    precision_top = default_top / k
    recall_top = default_top / max(y_true.sum(), 1)
    lift_top = precision_top / base if base > 0 else np.nan
    label = int(pct * 100)
    return {
        f"top_{label}pct_count": k,
        f"precision_at_top_{label}pct": precision_top,
        f"recall_at_top_{label}pct": recall_top,
        f"lift_at_top_{label}pct": lift_top,
    }

def calibration_summary(y_true, score, n_bins=10):
    tmp = pd.DataFrame({"y": np.asarray(y_true).astype(int), "score": np.asarray(score, dtype=float)})
    tmp["bin"] = pd.qcut(tmp["score"].rank(method="first"), q=n_bins, labels=False) + 1
    dec = tmp.groupby("bin").agg(
        customer_count=("y", "count"),
        observed_default_rate=("y", "mean"),
        mean_predicted_probability=("score", "mean"),
        observed_defaults=("y", "sum"),
    ).reset_index()
    dec["calibration_gap"] = dec["mean_predicted_probability"] - dec["observed_default_rate"]
    dec["abs_gap"] = dec["calibration_gap"].abs()
    ece = float((dec["abs_gap"] * dec["customer_count"] / dec["customer_count"].sum()).sum())
    mce = float(dec["abs_gap"].max())
    return ece, mce, dec

def threshold_metrics(y_true, score, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(score) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    return {
        "threshold": threshold, "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "predicted_positive_rate": float(y_pred.mean()),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "npv": npv,
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0,
    }

def find_best_thresholds(y_true, score):
    precision, recall, thresholds = precision_recall_curve(y_true, score)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_f1_idx = int(np.nanargmax(f1)) if len(f1) else 0
    fpr, tpr, roc_thresholds = roc_curve(y_true, score)
    j = tpr - fpr
    best_j_idx = int(np.nanargmax(j)) if len(j) else 0
    return {
        "best_f1_threshold": float(thresholds[best_f1_idx]) if len(thresholds) else 0.5,
        "best_f1_value": float(f1[best_f1_idx]) if len(f1) else np.nan,
        "best_youden_threshold": float(roc_thresholds[best_j_idx]) if len(roc_thresholds) else 0.5,
        "best_youden_j": float(j[best_j_idx]) if len(j) else np.nan,
    }

def evaluate_scores(model_name, model_group, y_true, score, runtime_seconds, probability_source, notes=""):
    y_true = np.asarray(y_true).astype(int)
    score = np.clip(np.asarray(score, dtype=float), 1e-7, 1 - 1e-7)
    best = find_best_thresholds(y_true, score)
    m05 = threshold_metrics(y_true, score, 0.5)
    mf1 = threshold_metrics(y_true, score, best["best_f1_threshold"])
    myj = threshold_metrics(y_true, score, best["best_youden_threshold"])
    ece, mce, _ = calibration_summary(y_true, score)
    row = {
        "model_name": model_name, "model_group": model_group, "status": "ok", "notes": notes,
        "probability_source": probability_source, "runtime_seconds": runtime_seconds,
        "n_val": len(y_true), "event_rate_val": float(y_true.mean()),
        "roc_auc": roc_auc_score(y_true, score),
        "gini": 2 * roc_auc_score(y_true, score) - 1,
        "pr_auc_average_precision": average_precision_score(y_true, score),
        "log_loss": log_loss(y_true, score, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, score),
        "ks_statistic": ks_statistic(y_true, score),
        "ece_calibration_error": ece, "mce_calibration_error": mce,
        **best,
    }
    for k, v in m05.items(): row[f"at_0_5_{k}"] = v
    for k, v in mf1.items(): row[f"at_best_f1_{k}"] = v
    for k, v in myj.items(): row[f"at_youden_{k}"] = v
    for pct in [0.05, 0.10, 0.20, 0.30]:
        row.update(top_percentile_metrics(y_true, score, pct))
    return row


In [ ]:
# ============================================================
# 5. Preprocessors and Model Registry
# ============================================================
def make_ohe_preprocessor(min_frequency=50):
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=min_frequency, sparse_output=True)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", min_frequency=min_frequency, sparse=True)
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("ohe", ohe),
        ]), cat_cols),
    ], remainder="drop", sparse_threshold=0.3)

def make_compact_preprocessor():
    return Pipeline([
        ("preprocess", make_ohe_preprocessor(min_frequency=100)),
        ("select", SelectKBest(score_func=f_classif, k=TOP_FEATURES_FOR_COMPACT_MODELS)),
        ("svd", TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_STATE)),
        ("scaler", StandardScaler()),
    ])

ordinal_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), cat_cols),
], remainder="drop")

models = [
    {"name": "Dummy Baseline - prior probability", "group": "Baseline",
     "estimator": DummyClassifier(strategy="prior", random_state=RANDOM_STATE),
     "notes": "Null baseline; predicts class prior."},
    {"name": "Logistic Regression - OHE baseline", "group": "Linear baseline",
     "estimator": Pipeline([("preprocess", make_ohe_preprocessor()), ("scaler", MaxAbsScaler()),
                            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="saga", n_jobs=-1, random_state=RANDOM_STATE))]),
     "notes": "Interpretable baseline, class_weight balanced."},
    {"name": "Decision Tree Classifier", "group": "Tree baseline",
     "estimator": Pipeline([("preprocess", make_ohe_preprocessor()),
                            ("clf", DecisionTreeClassifier(max_depth=6, min_samples_leaf=500, class_weight="balanced", random_state=RANDOM_STATE))]),
     "notes": "Simple tree; regularized to reduce overfit."},
    {"name": "Random Forest Classifier", "group": "Bagging tree",
     "estimator": Pipeline([("preprocess", make_ohe_preprocessor()),
                            ("clf", RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_leaf=100, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))]),
     "notes": "Bagging ensemble; stronger than single tree."},
    {"name": "Gradient Boosting Classifier - HistGB", "group": "Boosting sklearn",
     "estimator": Pipeline([("preprocess", ordinal_preprocessor),
                            ("clf", HistGradientBoostingClassifier(max_iter=500, learning_rate=0.04, max_leaf_nodes=31, l2_regularization=1.0, early_stopping=True, random_state=RANDOM_STATE))]),
     "notes": "Scalable sklearn gradient boosting for full data."},
    {"name": "Naive Bayes - Gaussian compact", "group": "Probabilistic baseline",
     "estimator": Pipeline([("compact", make_compact_preprocessor()), ("clf", GaussianNB())]),
     "notes": "Compact SVD features; NB independence assumption is unrealistic but useful baseline."},
]

if RUN_EXPENSIVE_MODELS:
    svm_pipe = Pipeline([("compact", make_compact_preprocessor()),
                         ("clf", LinearSVC(C=0.5, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))])
    models += [
        {"name": "Support Vector Machine - LinearSVC calibrated", "group": "SVM",
         "estimator": CalibratedClassifierCV(svm_pipe, method="sigmoid", cv=3),
         "notes": "Scalable linear SVM + calibration. RBF SVM is not practical on full data."},
        {"name": "K-Nearest Neighbors - compact SVD", "group": "Instance based",
         "estimator": Pipeline([("compact", make_compact_preprocessor()),
                                ("clf", KNeighborsClassifier(n_neighbors=101, weights="distance", n_jobs=-1))]),
         "notes": "Full rows with compact SVD representation."},
        {"name": "Neural Network - MLP compact", "group": "Neural Network",
         "estimator": Pipeline([("compact", make_compact_preprocessor()),
                                ("clf", MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-4, batch_size=2048, learning_rate_init=1e-3, max_iter=50, early_stopping=True, random_state=RANDOM_STATE))]),
         "notes": "Basic dense NN using compact features."},
    ]

print("Basic model count:", len(models))


In [ ]:
# ============================================================
# 6. Native Boosting Data and Research Recipes
# ============================================================
def prepare_lgbm_frame(df):
    out = df.copy()
    for c in cat_cols:
        if c in out.columns:
            out[c] = out[c].astype("category")
    return out

def prepare_catboost_frame(df):
    out = df.copy()
    for c in cat_cols:
        if c in out.columns:
            out[c] = out[c].astype("object").where(out[c].notna(), "Missing").astype(str)
    return out

X_train_lgb = prepare_lgbm_frame(X_train)
X_val_lgb = prepare_lgbm_frame(X_val)
X_test_lgb = prepare_lgbm_frame(X_test_final)

print("Fitting ordinal preprocessor...")
X_train_ord = ordinal_preprocessor.fit_transform(X_train)
X_val_ord = ordinal_preprocessor.transform(X_val)
X_test_ord = ordinal_preprocessor.transform(X_test_final)

X_train_cat = prepare_catboost_frame(X_train)
X_val_cat = prepare_catboost_frame(X_val)
X_test_cat = prepare_catboost_frame(X_test_final)
cat_feature_indices = [X_train_cat.columns.get_loc(c) for c in cat_cols if c in X_train_cat.columns]

scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
print("scale_pos_weight:", scale_pos_weight)

def lgbm_common():
    return dict(objective="binary", metric="auc", n_estimators=5000, learning_rate=0.03, n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1)

boosting_models = [
    {"name": "LightGBM - balanced baseline", "group": "LightGBM", "framework": "lightgbm",
     "params": dict(**lgbm_common(), num_leaves=31, max_depth=-1, colsample_bytree=0.7, subsample=0.9, reg_alpha=0.1, reg_lambda=1.0, min_child_samples=80, class_weight="balanced"),
     "notes": "Strong tabular baseline with class_weight balanced."},
    {"name": "XGBoost - hist baseline", "group": "XGBoost", "framework": "xgboost",
     "params": dict(n_estimators=5000, learning_rate=0.03, max_depth=4, min_child_weight=20, subsample=0.9, colsample_bytree=0.35, reg_alpha=3.0, reg_lambda=5.0, objective="binary:logistic", eval_metric="auc", tree_method="hist", scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, n_jobs=-1),
     "notes": "Regularized XGBoost for tabular credit risk."},
    {"name": "CatBoost - GPU/CPU baseline", "group": "CatBoost", "framework": "catboost",
     "params": dict(iterations=5000, learning_rate=0.03, depth=6, l2_leaf_reg=8, loss_function="Logloss", eval_metric="AUC", random_seed=RANDOM_STATE, auto_class_weights="Balanced", verbose=300, allow_writing_files=False),
     "notes": "Handles categorical variables natively."},
]

if RUN_RESEARCH_LGBM_RECIPES:
    boosting_models += [
        {"name": "LightGBM NoxMoon-style 1", "group": "Research recipe", "framework": "lightgbm",
         "params": dict(**lgbm_common(), num_leaves=26, max_depth=4, colsample_bytree=0.30, subsample=0.9320, reg_alpha=4.8299, reg_lambda=3.6335, min_split_gain=0.0068, min_child_weight=9.8138, class_weight={0: 1, 1: 1.0122}),
         "notes": "NoxMoon public recipe: shallow tree, low feature fraction, regularized."},
        {"name": "LightGBM NoxMoon-style 2", "group": "Research recipe", "framework": "lightgbm",
         "params": dict(**lgbm_common(), num_leaves=26, max_depth=4, colsample_bytree=0.28, subsample=0.95, reg_alpha=4.8299, reg_lambda=3.6335, min_split_gain=0.005, min_child_weight=40, class_weight={0: 1, 1: 1}),
         "notes": "NoxMoon variation with stronger min_child_weight."},
        {"name": "LightGBM NoxMoon-style 3", "group": "Research recipe", "framework": "lightgbm",
         "params": dict(**lgbm_common(), num_leaves=26, max_depth=4, colsample_bytree=0.30, subsample=0.9320, reg_alpha=4.8299, reg_lambda=20, min_split_gain=0.0068, min_child_weight=9.8138, class_weight={0: 1, 1: 1.0122}),
         "notes": "NoxMoon variation with stronger L2."},
        {"name": "LightGBM oskird-style", "group": "Research recipe", "framework": "lightgbm",
         "params": dict(objective="binary", metric="auc", n_estimators=15000, learning_rate=0.01, num_leaves=36, colsample_bytree=0.10, subsample=0.90, reg_alpha=3, reg_lambda=1, min_split_gain=0.01, max_depth=7, n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1),
         "notes": "oskird top-3%-style: low learning rate, very low feature fraction."},
    ]

print("Boosting model count:", len(boosting_models))


In [ ]:
# ============================================================
# 7. Train and Evaluate All Holdout Models
# ============================================================
results, val_predictions, test_predictions, trained_models = [], {}, {}, {}

def register_result(model_name, model_group, y_true, val_score, test_score, runtime, source, notes, model_obj=None):
    row = evaluate_scores(model_name, model_group, y_true, val_score, runtime, source, notes)
    results.append(row)
    val_predictions[model_name] = np.asarray(val_score, dtype=float)
    test_predictions[model_name] = np.asarray(test_score, dtype=float)
    if model_obj is not None:
        trained_models[model_name] = model_obj
    pd.DataFrame(results).sort_values("roc_auc", ascending=False).to_csv(OUTPUT_DIR / "ml_model_comparison_live.csv", index=False)
    print(f"DONE {model_name}: ROC-AUC={row['roc_auc']:.6f}, PR-AUC={row['pr_auc_average_precision']:.6f}, Lift@10={row['lift_at_top_10pct']:.3f}")

for model_def in models:
    name = model_def["name"]
    print("\n" + "=" * 90 + f"\nTraining: {name}\n" + "=" * 90)
    t0 = time.time()
    try:
        model = model_def["estimator"]
        model.fit(X_train, y_train)
        val_score, src = safe_proba_from_model(model, X_val)
        test_score, _ = safe_proba_from_model(model, X_test_final)
        register_result(name, model_def["group"], y_val, val_score, test_score, time.time() - t0, src, model_def.get("notes", ""), model)
    except Exception as e:
        print("FAILED:", name, e)
        results.append({"model_name": name, "model_group": model_def["group"], "status": "failed", "error": str(e), "runtime_seconds": time.time() - t0, "notes": model_def.get("notes", "")})
    gc.collect()

def fit_predict_lightgbm(model_def):
    params = model_def["params"].copy()
    first = params.copy()
    if USE_GPU:
        first["device"] = "gpu"
    try:
        model = LGBMClassifier(**first)
        model.fit(
            X_train_lgb, y_train,
            eval_set=[(X_val_lgb, y_val)],
            eval_metric="auc",
            callbacks=[lgb_early_stopping(EARLY_STOPPING_ROUNDS), lgb_log_evaluation(300)]
        )
    except Exception as e:
        print("LightGBM first attempt failed, retry CPU:", e)
        params.pop("device", None)
        model = LGBMClassifier(**params)
        model.fit(
            X_train_lgb, y_train,
            eval_set=[(X_val_lgb, y_val)],
            eval_metric="auc",
            callbacks=[lgb_early_stopping(EARLY_STOPPING_ROUNDS), lgb_log_evaluation(300)]
        )
    return model, model.predict_proba(X_val_lgb)[:, 1], model.predict_proba(X_test_lgb)[:, 1], "predict_proba"

def fit_predict_xgboost(model_def):
    params = model_def["params"].copy()
    first = params.copy()
    if USE_GPU:
        first["device"] = "cuda"
    try:
        model = XGBClassifier(**first)
        try:
            model.fit(X_train_ord, y_train, eval_set=[(X_val_ord, y_val)], verbose=300, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        except TypeError:
            model.set_params(early_stopping_rounds=EARLY_STOPPING_ROUNDS)
            model.fit(X_train_ord, y_train, eval_set=[(X_val_ord, y_val)], verbose=300)
    except Exception as e:
        print("XGBoost first attempt failed, retry CPU:", e)
        params.pop("device", None)
        model = XGBClassifier(**params)
        try:
            model.fit(X_train_ord, y_train, eval_set=[(X_val_ord, y_val)], verbose=300, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        except TypeError:
            model.set_params(early_stopping_rounds=EARLY_STOPPING_ROUNDS)
            model.fit(X_train_ord, y_train, eval_set=[(X_val_ord, y_val)], verbose=300)
    return model, model.predict_proba(X_val_ord)[:, 1], model.predict_proba(X_test_ord)[:, 1], "predict_proba"

def fit_predict_catboost(model_def):
    params = model_def["params"].copy()
    first = params.copy()
    if USE_GPU:
        first["task_type"] = "GPU"
    train_pool = Pool(X_train_cat, y_train, cat_features=cat_feature_indices)
    val_pool = Pool(X_val_cat, y_val, cat_features=cat_feature_indices)
    test_pool = Pool(X_test_cat, cat_features=cat_feature_indices)
    try:
        model = CatBoostClassifier(**first)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True)
    except Exception as e:
        print("CatBoost first attempt failed, retry CPU:", e)
        params.pop("task_type", None)
        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=EARLY_STOPPING_ROUNDS, use_best_model=True)
    return model, model.predict_proba(val_pool)[:, 1], model.predict_proba(test_pool)[:, 1], "predict_proba"

for model_def in boosting_models:
    name = model_def["name"]
    print("\n" + "=" * 90 + f"\nTraining: {name}\n" + "=" * 90)
    t0 = time.time()
    try:
        if model_def["framework"] == "lightgbm":
            model, val_score, test_score, src = fit_predict_lightgbm(model_def)
        elif model_def["framework"] == "xgboost":
            model, val_score, test_score, src = fit_predict_xgboost(model_def)
        elif model_def["framework"] == "catboost":
            model, val_score, test_score, src = fit_predict_catboost(model_def)
        register_result(name, model_def["group"], y_val, val_score, test_score, time.time() - t0, src, model_def.get("notes", ""), model)
    except Exception as e:
        print("FAILED:", name, e)
        results.append({"model_name": name, "model_group": model_def["group"], "status": "failed", "error": str(e), "runtime_seconds": time.time() - t0, "notes": model_def.get("notes", "")})
    gc.collect()

all_results = pd.DataFrame(results).sort_values("roc_auc", ascending=False, na_position="last")
display(all_results.head(30))
all_results.to_csv(OUTPUT_DIR / "03_all_model_results_holdout.csv", index=False)


In [ ]:
# ============================================================
# 8. WOE Logistic Regression and Scorecard
# ============================================================
def cap_rare_categories(train_df, valid_df, test_df, cols, min_count=100):
    tr, va, te = train_df.copy(), valid_df.copy(), test_df.copy()
    for c in cols:
        vc = tr[c].astype(str).value_counts()
        keep = set(vc[vc >= min_count].index)
        tr[c] = tr[c].astype(str).where(tr[c].astype(str).isin(keep), "Other rare")
        va[c] = va[c].astype(str).where(va[c].astype(str).isin(keep), "Other rare")
        te[c] = te[c].astype(str).where(te[c].astype(str).isin(keep), "Other rare")
    return tr, va, te

def make_woe_binned_frame(train_df, valid_df, test_df, numeric_cols, cat_cols, n_bins=10):
    train_out, valid_out, test_out = pd.DataFrame(index=train_df.index), pd.DataFrame(index=valid_df.index), pd.DataFrame(index=test_df.index)
    for c in numeric_cols:
        try:
            q = train_df[c].quantile(np.linspace(0, 1, n_bins + 1)).drop_duplicates().values
            if len(q) <= 2:
                raise ValueError("not enough bins")
            train_out[c] = pd.cut(train_df[c], bins=q, include_lowest=True, duplicates="drop").astype(str).fillna("Missing")
            valid_out[c] = pd.cut(valid_df[c], bins=q, include_lowest=True, duplicates="drop").astype(str).fillna("Missing")
            test_out[c] = pd.cut(test_df[c], bins=q, include_lowest=True, duplicates="drop").astype(str).fillna("Missing")
        except Exception:
            train_out[c] = train_df[c].fillna("Missing").astype(str)
            valid_out[c] = valid_df[c].fillna("Missing").astype(str)
            test_out[c] = test_df[c].fillna("Missing").astype(str)
    for c in cat_cols:
        train_out[c] = train_df[c].fillna("Missing").astype(str)
        valid_out[c] = valid_df[c].fillna("Missing").astype(str)
        test_out[c] = test_df[c].fillna("Missing").astype(str)
    return train_out, valid_out, test_out

t0 = time.time()
try:
    Xtr_cap, Xv_cap, Xt_cap = cap_rare_categories(X_train, X_val, X_test_final, cat_cols)
    Xtr_woe, Xv_woe, Xt_woe = make_woe_binned_frame(Xtr_cap, Xv_cap, Xt_cap, num_cols, cat_cols)
    woe_model = Pipeline([
        ("woe", ce.WOEEncoder(cols=Xtr_woe.columns.tolist(), regularization=0.5)),
        ("lr", LogisticRegression(max_iter=2000, solver="lbfgs", class_weight="balanced")),
    ])
    woe_model.fit(Xtr_woe, y_train)
    val_score, src = safe_proba_from_model(woe_model, Xv_woe)
    test_score, _ = safe_proba_from_model(woe_model, Xt_woe)
    register_result("WOE Logistic Regression", "Credit scoring", y_val, val_score, test_score, time.time() - t0, src, "WOE-binned variables + Logistic Regression.", woe_model)
    register_result("Logistic Scorecard Model", "Credit scoring", y_val, val_score, test_score, time.time() - t0, src, "Same WOE-LR predictions interpreted as scorecard model.", woe_model)
except Exception as e:
    print("WOE/Scorecard failed:", e)
    results.append({"model_name": "WOE Logistic Regression", "model_group": "Credit scoring", "status": "failed", "error": str(e)})
    results.append({"model_name": "Logistic Scorecard Model", "model_group": "Credit scoring", "status": "failed", "error": str(e)})

pd.DataFrame(results).sort_values("roc_auc", ascending=False, na_position="last").to_csv(OUTPUT_DIR / "03_all_model_results_holdout.csv", index=False)


In [ ]:
# ============================================================
# 9. Applicability Table for Models Not Directly Valid Here
# ============================================================
applicability = pd.DataFrame([
    {"model_family": "Bayesian Logistic Regression", "run_status": "documented_not_run_by_default", "reason": "Full Bayesian sampling is too slow for full-data benchmark; classical Logit already covered in Step 6."},
    {"model_family": "Multilevel Logistic Regression", "run_status": "not_applicable_current_table", "reason": "Needs clear random-effect hierarchy such as branch/region; final table is customer-level."},
    {"model_family": "Discrete-Time Hazard Model", "run_status": "not_applicable_current_target", "reason": "Needs person-period/event-timing target; current TARGET is binary default."},
    {"model_family": "Cox Proportional Hazards Model", "run_status": "not_applicable_current_target", "reason": "Needs time-to-event and censoring; not available in final analytical table."},
    {"model_family": "Nonlinear RBF SVM", "run_status": "replaced_by_scalable_linear_svm", "reason": "RBF SVM does not scale to 300k+ rows/high-dimensional data; calibrated LinearSVC is used."},
])
display(applicability)
applicability.to_csv(OUTPUT_DIR / "04_model_applicability_notes.csv", index=False)


In [ ]:
# ============================================================
# 10. Ensemble and Optional CV for Top LightGBM Recipes
# ============================================================
ok_results = pd.DataFrame(results)
ok_results = ok_results[ok_results["status"].eq("ok")].sort_values(["roc_auc", "pr_auc_average_precision"], ascending=False)
display(ok_results[["model_name", "model_group", "roc_auc", "pr_auc_average_precision", "log_loss", "brier_score", "lift_at_top_10pct", "runtime_seconds"]].head(20))

top_names = ok_results.head(min(5, len(ok_results)))["model_name"].tolist()
if len(top_names) >= 2:
    val_blend = np.mean([val_predictions[n] for n in top_names], axis=0)
    test_blend = np.mean([test_predictions[n] for n in top_names], axis=0)
    register_result(f"Blend top {len(top_names)} average", "Ensemble", y_val, val_blend, test_blend, 0.0, "average_probabilities", "Simple average of top holdout models.")
    weights = ok_results.head(len(top_names))["roc_auc"].values - 0.5
    weights = np.maximum(weights, 0) / np.maximum(np.maximum(weights, 0).sum(), 1e-12)
    val_wblend = np.average([val_predictions[n] for n in top_names], axis=0, weights=weights)
    test_wblend = np.average([test_predictions[n] for n in top_names], axis=0, weights=weights)
    register_result(f"Blend top {len(top_names)} weighted by ROC-AUC", "Ensemble", y_val, val_wblend, test_wblend, 0.0, "weighted_average_probabilities", "Weighted average using ROC-AUC above random.")

def cv_lightgbm_recipe(model_def, X_full, y_full, X_test_full):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y_full))
    test_pred = np.zeros(len(X_test_full))
    fold_rows = []
    params = model_def["params"].copy()
    params.pop("device", None)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full), 1):
        print(f"{model_def['name']} fold {fold}/{N_FOLDS}")
        Xtr, Xva = prepare_lgbm_frame(X_full.iloc[tr_idx]), prepare_lgbm_frame(X_full.iloc[va_idx])
        ytr, yva = y_full.iloc[tr_idx], y_full.iloc[va_idx]
        model = LGBMClassifier(**params)
        model.fit(
            Xtr, ytr,
            eval_set=[(Xva, yva)],
            eval_metric="auc",
            callbacks=[lgb_early_stopping(EARLY_STOPPING_ROUNDS), lgb_log_evaluation(300)]
        )
        pred = model.predict_proba(Xva)[:, 1]
        oof[va_idx] = pred
        test_pred += model.predict_proba(prepare_lgbm_frame(X_test_full))[:, 1] / N_FOLDS
        fold_rows.append({"model_name": model_def["name"], "fold": fold, "roc_auc": roc_auc_score(yva, pred), "pr_auc": average_precision_score(yva, pred)})
        del model, Xtr, Xva
        gc.collect()
    return oof, test_pred, pd.DataFrame(fold_rows)

cv_results, cv_test_predictions = [], {}
if RUN_CV_TOP_MODELS:
    top_lgbm_recipes = [m for m in boosting_models if m["framework"] == "lightgbm"][:3]
    for recipe in top_lgbm_recipes:
        t0 = time.time()
        try:
            oof, test_pred, fold_df = cv_lightgbm_recipe(recipe, X, y, X_test_final)
            fold_df.to_csv(OUTPUT_DIR / f"cv_folds_{recipe['name'].replace(' ', '_')}.csv", index=False)
            row = evaluate_scores(recipe["name"] + " - 5Fold OOF", "CV top model", y, oof, time.time() - t0, "5fold_oof_predict_proba", recipe.get("notes", ""))
            cv_results.append(row)
            cv_test_predictions[recipe["name"] + " - 5Fold OOF"] = test_pred
            print(f"CV DONE {recipe['name']}: ROC-AUC={row['roc_auc']:.6f}")
        except Exception as e:
            print("CV failed:", recipe["name"], e)
            cv_results.append({"model_name": recipe["name"] + " - 5Fold OOF", "model_group": "CV top model", "status": "failed", "error": str(e)})

cv_results_df = pd.DataFrame(cv_results).sort_values("roc_auc", ascending=False, na_position="last") if cv_results else pd.DataFrame()
if not cv_results_df.empty:
    display(cv_results_df)
    cv_results_df.to_csv(OUTPUT_DIR / "06_cv_top_model_results.csv", index=False)


In [ ]:
# ============================================================
# 11. Final Tables, Charts, SHAP, Submission
# ============================================================
final_results = pd.concat([pd.DataFrame(results), cv_results_df if "cv_results_df" in globals() and not cv_results_df.empty else pd.DataFrame()], ignore_index=True, sort=False)
final_ok = final_results[final_results["status"].eq("ok")].sort_values(["roc_auc", "pr_auc_average_precision", "lift_at_top_10pct"], ascending=False)
final_results.to_csv(OUTPUT_DIR / "07_final_all_model_comparison.csv", index=False)
final_ok.to_csv(OUTPUT_DIR / "08_final_ok_model_comparison.csv", index=False)

cols = ["model_name", "model_group", "roc_auc", "pr_auc_average_precision", "log_loss", "brier_score", "ks_statistic", "lift_at_top_10pct", "precision_at_top_10pct", "recall_at_top_10pct", "at_0_5_precision", "at_0_5_recall_sensitivity", "best_f1_threshold", "best_f1_value", "runtime_seconds", "notes"]
display(final_ok[[c for c in cols if c in final_ok.columns]].head(40))

plt.figure(figsize=(12, 7))
plot_df = final_ok.head(20).sort_values("roc_auc")
plt.barh(plot_df["model_name"], plot_df["roc_auc"], color="#6C4CF6")
plt.xlabel("ROC-AUC")
plt.title("Model Comparison by ROC-AUC")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison_roc_auc.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(12, 7))
plot_df = final_ok.head(20).sort_values("pr_auc_average_precision")
plt.barh(plot_df["model_name"], plot_df["pr_auc_average_precision"], color="#FF5C7A")
plt.xlabel("PR-AUC / Average Precision")
plt.title("Model Comparison by PR-AUC")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison_pr_auc.png", dpi=160, bbox_inches="tight")
plt.show()

top_holdout = pd.DataFrame(results)
top_holdout = top_holdout[top_holdout["status"].eq("ok")].sort_values(["roc_auc", "pr_auc_average_precision"], ascending=False).head(5)["model_name"].tolist()

plt.figure(figsize=(8, 7))
for name in top_holdout:
    if name in val_predictions:
        fpr, tpr, _ = roc_curve(y_val, val_predictions[name])
        plt.plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_val, val_predictions[name]):.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.title("ROC Curve - Top Holdout Models")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve_top_models.png", dpi=160, bbox_inches="tight")
plt.show()

calibration_tables = []
for name in top_holdout:
    if name in val_predictions:
        _, _, dec = calibration_summary(y_val, val_predictions[name])
        dec["model_name"] = name
        calibration_tables.append(dec)
if calibration_tables:
    calibration_df = pd.concat(calibration_tables, ignore_index=True)
    calibration_df.to_csv(OUTPUT_DIR / "09_calibration_deciles_top_models.csv", index=False)

best_holdout_name = top_holdout[0] if top_holdout else None
if best_holdout_name in trained_models:
    model = trained_models[best_holdout_name]
    try:
        if hasattr(model, "feature_importances_"):
            feat_names = X_train_lgb.columns.tolist() if "LightGBM" in best_holdout_name else [f"f{i}" for i in range(len(model.feature_importances_))]
            fi = pd.DataFrame({"feature": feat_names[:len(model.feature_importances_)], "importance": model.feature_importances_}).sort_values("importance", ascending=False)
        elif hasattr(model, "get_feature_importance"):
            fi = pd.DataFrame({"feature": X_train_cat.columns, "importance": model.get_feature_importance()}).sort_values("importance", ascending=False)
        else:
            fi = pd.DataFrame()
        if not fi.empty:
            fi.to_csv(OUTPUT_DIR / "10_feature_importance_best_model.csv", index=False)
            display(fi.head(30))
            plt.figure(figsize=(10, 8))
            top_fi = fi.head(30).sort_values("importance")
            plt.barh(top_fi["feature"], top_fi["importance"], color="#6C4CF6")
            plt.title(f"Feature Importance - {best_holdout_name}")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "feature_importance_best_model.png", dpi=160, bbox_inches="tight")
            plt.show()
    except Exception as e:
        print("Feature importance failed:", e)

    if RUN_SHAP:
        try:
            shap_sample = min(5000, len(X_val))
            idx = np.random.RandomState(RANDOM_STATE).choice(np.arange(len(X_val)), size=shap_sample, replace=False)
            if "LightGBM" in best_holdout_name:
                shap_X = X_val_lgb.iloc[idx]
                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(shap_X)
                if isinstance(shap_values, list):
                    shap_values = shap_values[1]
                shap.summary_plot(shap_values, shap_X, show=False, max_display=30)
                plt.tight_layout()
                plt.savefig(OUTPUT_DIR / "shap_summary_best_model.png", dpi=160, bbox_inches="tight")
                plt.show()
            elif "CatBoost" in best_holdout_name:
                shap_X = X_val_cat.iloc[idx]
                pool = Pool(shap_X, cat_features=cat_feature_indices)
                shap_values = model.get_feature_importance(pool, type="ShapValues")[:, :-1]
                shap.summary_plot(shap_values, shap_X, show=False, max_display=30)
                plt.tight_layout()
                plt.savefig(OUTPUT_DIR / "shap_summary_best_model.png", dpi=160, bbox_inches="tight")
                plt.show()
        except Exception as e:
            print("SHAP failed:", e)

submission_candidates = {}
submission_candidates.update(test_predictions)
if "cv_test_predictions" in globals():
    submission_candidates.update(cv_test_predictions)

submission_model = None
for name in final_ok["model_name"].tolist():
    if name in submission_candidates:
        submission_model = name
        break
if submission_model is None:
    raise RuntimeError("No submission prediction available.")

pred = np.clip(submission_candidates[submission_model], 0, 1)
if SAMPLE_SUB_PATH is not None:
    submission = pd.read_csv(SAMPLE_SUB_PATH)
    submission["TARGET"] = pred
else:
    submission = pd.DataFrame({ID_COL: ids_test, "TARGET": pred})
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print("Submission model:", submission_model)
display(submission.head())


In [ ]:
# ============================================================
# 12. Research Sources and Presentation Summary
# ============================================================
research_sources = pd.DataFrame([
    {"source": "NoxMoon Home Credit solution, Rank 17/7198", "method_to_borrow": "LightGBM shallow regularized recipes, XGBoost, weighted average, GP features", "used_in_notebook": "NoxMoon-style LightGBM 1/2/3 + ensemble logic", "url": "https://github.com/NoxMoon/home-credit-default-risk"},
    {"source": "oskird Kaggle Home Credit Top 3% solution", "method_to_borrow": "LightGBM low learning rate, low feature fraction, Bayesian optimization, stacking", "used_in_notebook": "oskird-style LightGBM + WOE/scorecard comparison + ensemble", "url": "https://github.com/oskird/Kaggle-Home-Credit-Default-Risk-Solution"},
    {"source": "Home Credit 2nd place solution deck", "method_to_borrow": "Interest-rate features, time-aware FE, LightGBM/DART, NN sequence models, ensemble", "used_in_notebook": "Regularized boosting, low feature fraction direction, ensemble mindset", "url": "https://speakerdeck.com/hoxomaxwell/home-credit-default-risk-2nd-place-solutions"},
])
display(research_sources)
research_sources.to_csv(OUTPUT_DIR / "11_research_sources_used.csv", index=False)

presentation_cols = ["model_name", "model_group", "roc_auc", "pr_auc_average_precision", "log_loss", "brier_score", "ks_statistic", "lift_at_top_10pct", "precision_at_top_10pct", "recall_at_top_10pct", "at_0_5_precision", "at_0_5_recall_sensitivity", "best_f1_threshold", "best_f1_value", "runtime_seconds", "notes"]
presentation = final_ok[[c for c in presentation_cols if c in final_ok.columns]].copy()
presentation.to_csv(OUTPUT_DIR / "12_presentation_model_summary.csv", index=False)
display(presentation.head(30))

print("All outputs saved to:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)
